<a href="https://colab.research.google.com/github/rohanraaj2/Spoken-and-Natural-Language-Understanding/blob/main/Assignment%202/Assignment_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Assignmnet 2 (100 points)

**Name:** Rohan Raj <br>
**Email:** ror0011@thi.de <br>
**Group:** A <br>
**Hours spend *(optional)* :** <br>

### SMS Spam Detection

<p>You are hired as an AI expert in the development department of a telecommunications company. The first thing on your orientation plan is a small project that your boss has assigned you for the following given situation. Your supervisor has given away his private cell phone number on too many websites and is now complaining about daily spam SMS. Therefore, it is your job to write a spam detector in Python. </p>

<p>In doing so, you need to use a Naive Bayes classifier that can handle both bag-of-words (BoW) and tf-idf features as input. For the evaluation of your spam detector, an SMS collection is available as a dataset - this has yet to be suitably split into train and test data. To keep the costs as low as possible and to avoid problems with copyrights, your boss insists on a new development with Python.</p>

<p>Include a short description of the data preprocessing steps, method, experiment design, hyper-parameters, and evaluation metric. Also, document your findings, drawbacks, and potential improvements.</p>

<p>Note: You need to implement the bag-of-words (BoW) and tf-idf feature extractor from scratch. You can use existing python libraries for other tasks.</p>

**Dataset and Resources**

* SMS Spam Collection Dataset: https://archive.ics.uci.edu/dataset/228/sms+spam+collection

In [29]:
'''
1) Data Preprocessing:
- I will read through the entire file and then create a list for spam and ham. I will append each spam message into spam list and ham message into ham list. This will be known by reading the first word in each line.
- Then clean the data by taking into account the punctuation, non alphabets, etc.
- Then I will lowercase and then tokenize the words
- Later I will split the data into training and test data. It will be split 80:20.

2) Method:
- I will use the python's read file which will store the dataset into a variable as string
- After preprocessing, I would create a dictionary of spam and ham words, messages as well as list containing all words. All three lists with their count of words
- in the tf-idf function, i will calculate the tf-idf scores by first calculating tf scores and then idf scores and then using the formula to calculate the tf-idf score

3) Experiment Design:
- I split the dataset in the ratio of 80% training dataset while rest 20% in test dataset
- It divided in chronological order. First 80% of messages were used for training while the remaining 20% were used for test dataset

4) Hyper-parameters:
- I applied Laplace smoothing to avoid getting 0 as numerator or denominator in calculations which would result the probability to 0 and make the whole product to 0 if used in the product.

5) Evaluation Metric:
- I will implement Naive Bayes classifier from scratch to fit our implemented instead of converting our implemented to pass to sk-learn method
- Then we use Naive Bayes classifier to calculate the probabilities of the message being spam and ham
- We will use accuracy as evaluation metric as it's better to let a spam message through than to let a ham message marked as spam. This was suggested by AI

6) Findings:
- Our model achieves 98.6% accuracy on both training and test data set.

7) Drawbacks:
- While good, even the little inaccuracy could cause issues
- Words can be censored to avoid spam detection. This was suggested by AI

8) Potential improvements:
- Have more and distinct sample dataset
- Preprocessing data to simple words (running/runs to run, etc). This was suggested by AI

'''
import re, math
from collections import Counter

def bag_of_words(documents):
  ## Complete the function
  spam_messages = documents['spam']
  ham_messages = documents['ham']

  # We use counter to create a dictionary of words with their respective count/number of times they are used.
  spam_words = Counter() # Only
  ham_words = Counter()
  vocabulary = Counter()

  for message in spam_messages:
    spam_words.update(message) # takes words from a message, stores them in a dictionary with their count. The .update(message) appends the count. It takes the count not only from one message but across all messages
    vocabulary.update(message)
  for message in ham_messages:
    ham_words.update(message)
    vocabulary.update(message)

  spam_message_count = len(documents['spam'])
  ham_message_count = len(documents['ham'])

  return spam_message_count, spam_words, ham_message_count, ham_words, vocabulary

message_word_ratio = {}
idf = {}
tf_idf_message_score = {}

def tf_idf(documents, laplace_constant, is_training = False):
  ## Complete the function
  global message_word_ratio, idf, tf_idf_message_score
  tf_idf_message_score = {}
  word_appearance = {}
  relative_frequencies = {}
  tf_idf_score = {}

  for doc_type in documents: # Spam or Ham
    for message in documents[doc_type]:
      tf = Counter(message) # For each message, takes the words in it and stores in a dictionary with their count in that message.
      unique_words = set(tf) # Counts each word as once.

      for word in unique_words: # We are checking whether the word appears in the message, not how many times.
        if word not in word_appearance:
          word_appearance[word] = 1
        else:
          word_appearance[word] += 1

  messages_count = len(documents['spam']) + len(documents['ham']) # Total number of messages

  for doc_type in documents: # Spam or Ham
    for idx, message in enumerate(documents[doc_type]): # Takes index (message number) and the message itself
      tf = Counter(message)
      tf_idf_word_score = {} # Dictionary to check word score for the current message
      for word in tf:
        relative_frequency = tf[word] / len(message) # Frequency of word divided by length of message. This is to ensure that length of message does not create a bias
        if is_training == True: # If training set is being used
          message_word_ratio[word] = messages_count / word_appearance[word] # We create dataset to store idf values
        idf[word] = math.log(message_word_ratio.get(word, 0) + laplace_constant) # If the data does not exist, we take 0 value and add laplace constant. This is useful for test dataset for unseen values
        tf_idf_word_score[word] = relative_frequency * idf[word] # TF-IDF score for each word in that message
      tf_idf_message_score[(doc_type, idx)] = tf_idf_word_score # Score of all words in that message

  return tf_idf_message_score

def naive_bayes_classifier(data, tf_idf_score_with_message, laplace_constant):
  correct = 0
  wrong = 0
  spam_message_count, spam_words, ham_message_count, ham_words, vocabulary = data
  total_spam_count = sum(spam_words.values()) # I used this since it also takes into account the count of words unlike len() that only considers unique words
  total_ham_count = sum(ham_words.values())
  vocab_size = len(vocabulary)

  # The mathematical formula was told by AI since I didn't remember
  prior_spam = math.log(spam_message_count / (spam_message_count + ham_message_count))
  prior_ham = math.log(ham_message_count / (spam_message_count + ham_message_count))
  for message_id in tf_idf_score_with_message:
    sum_of_tf_idfs = 0
    sum_of_spam_probs = 0
    sum_of_ham_probs = 0
    scores = tf_idf_score_with_message[message_id]

    for word in scores: # Smoothing is applied
      sum_of_spam_probs += math.log((spam_words[word] + laplace_constant) / (total_spam_count + vocab_size))
      sum_of_ham_probs += math.log((ham_words[word] + laplace_constant) / (total_ham_count + vocab_size))

    final_score_spam = sum_of_spam_probs + prior_spam
    final_score_ham = sum_of_ham_probs + prior_ham

    if final_score_spam > final_score_ham:
      label = 'spam'
    else:
      label = 'ham'
    # print ('Predicted:', label, 'Actual:', message_id[0])

    if label == message_id[0]:
      # print ('Correct Prediction')
      correct += 1
    else:
      # print ('Wrong Prediction')
      wrong += 1

  accuracy = correct / (correct + wrong)
  print ('Accuracy:', accuracy)

training_set = {'spam': [], 'ham': []}
test_set = {'spam': [], 'ham': []}

with open('SMSSpamCollection') as f:
  lines = f.readlines() # Takes file data and inputs as string in variable lines
  for i in range(len(lines)):
    categorized = lines[i].split('\t') # This separates the label and the message content
    lowercasetext = categorized[1].lower() # Data preprocessing
    token = re.findall(r"\w+", lowercasetext)
    if i <= int(len(lines) * 0.8): # First 80% of data moved to training dataset
      training_set[categorized[0]].append(token)
    else:
      test_set[categorized[0]].append(token) # Rest 20% to test dataset

  data = bag_of_words(training_set)
  idf_scores = tf_idf(training_set, 0.1, True) # Training dataset passed with laplace constant as 0.1 with flag as training dataset
  test_data = bag_of_words(test_set)
  test_idf_scores = tf_idf(test_set, 0.1) # Test dataset passed with laplace constant as 0.1 with flag as training dataset
  naive_bayes_classifier(data, idf_scores, 0.1) # Classifer to training dataset
  naive_bayes_classifier(data, test_idf_scores, 0.1) # Classifer to test dataset

## You can use sklearn or other python libraries for naive bayes classifier, evaluation metric, etc.  ##

Accuracy: 0.997085201793722
Accuracy: 0.9838420107719928


### Additional Experiments *(5 additional points - Optional)*

In this section, you can explore and document any additional experiments you conduct, such as trying different model architectures for classifiers, exploring hyperparameter tuning in more detail, or using different feature extraction methods.

<!-- Add comments about additional experiments here -->

In [30]:
# In this experiment, we will run the same model with different values of laplace constants

# Using laplace constant as 0.01
idf_scores = tf_idf(training_set, 0.01, True)
test_idf_scores = tf_idf(test_set, 0.01)
naive_bayes_classifier(data, idf_scores, 0.01)
naive_bayes_classifier(data, test_idf_scores, 0.01)

# Using laplace constant as 0.05
idf_scores = tf_idf(training_set, 0.05, True)
test_idf_scores = tf_idf(test_set, 0.05)
naive_bayes_classifier(data, idf_scores, 0.05)
naive_bayes_classifier(data, test_idf_scores, 0.05)

# Using laplace constant as 0.001
idf_scores = tf_idf(training_set, 0.001, True)
test_idf_scores = tf_idf(test_set, 0.001)
naive_bayes_classifier(data, idf_scores, 0.001)
naive_bayes_classifier(data, test_idf_scores, 0.001)

# Using laplace constant as 0.0005
idf_scores = tf_idf(training_set, 0.0005, True)
test_idf_scores = tf_idf(test_set, 0.0005)
naive_bayes_classifier(data, idf_scores, 0.0005)
naive_bayes_classifier(data, test_idf_scores, 0.0005)

print("Reducing the laplace constant generally increases the accuracy since there is less probability assigned to unseen data.")

Accuracy: 0.997982062780269
Accuracy: 0.9865350089766607
Accuracy: 0.9973094170403587
Accuracy: 0.9865350089766607
Accuracy: 0.9982062780269059
Accuracy: 0.9847396768402155
Accuracy: 0.9984304932735426
Accuracy: 0.9856373429084381
Reducing the laplace constant generally increases the accuracy since there is less probability assigned to unseen data.
